# Week 6 -- Function 2: GP + UCB Exploitation (2D)

In [1]:
# -- WEEKLY OBSERVATIONS LOG
import numpy as np

INITIAL_SIZE = 10
DATA_PATH_X  = '../data/function_2/initial_inputs.npy'
DATA_PATH_Y  = '../data/function_2/initial_outputs.npy'

weekly_log = [
    ([0.000000, 1.000000], 0.02486946982716038),   # W1: boundary dead
    ([1.000000, 0.326531], 0.14927005994804757),   # W2: BEST
    ([1.000000, 1.000000], 0.03643861312405127),   # W3: x2 too high
    ([0.998382, 0.000400], -0.0548095379957204),   # W4: x2 too low (NN went negative)
    ([0.001256, 0.001021], 0.0325117043072872),    # W5: GP EI+SVM -- dead corner
]

X_disk = np.load(DATA_PATH_X)
Y_disk = np.load(DATA_PATH_Y)

n_missing = (INITIAL_SIZE + len(weekly_log)) - len(Y_disk)
if n_missing > 0:
    new_entries = weekly_log[len(weekly_log) - n_missing:]
    for x_vals, y_val in new_entries:
        X_disk = np.vstack([X_disk, np.array([x_vals])])
        Y_disk = np.append(Y_disk, y_val)
    np.save(DATA_PATH_X, X_disk)
    np.save(DATA_PATH_Y, Y_disk)
    print(f'Synced {n_missing} new observation(s).')
else:
    print('Already up-to-date.')

print(f'X: {X_disk.shape} | Y: {Y_disk.shape}')
current_week = len(Y_disk) - INITIAL_SIZE + 1
print(f'Week {current_week} | {len(Y_disk)} total observations ({INITIAL_SIZE} initial + {len(Y_disk)-INITIAL_SIZE} added)')

Already up-to-date.
X: (15, 2) | Y: (15,)
Week 6 | 15 total observations (10 initial + 5 added)


In [2]:
# Cell 3: Load data and inspect -- Function 2: Noisy Chemical Process (2D), Maximise

X = np.load(DATA_PATH_X)
Y = np.load(DATA_PATH_Y)
n_dims = X.shape[1]

print(f'Input  shape : {X.shape}')
print(f'Output shape : {Y.shape}')
print()

# Sort by Y directly -- F2 values are in normal range, positive = better
pairs = sorted(zip(Y.tolist(), X.tolist()), reverse=True)
Y_sorted = [p[0] for p in pairs]
X_sorted = [p[1] for p in pairs]

print('=' * 72)
print('  All observations (sorted descending by Y)')
print('=' * 72)
for i, (y_val, x_val) in enumerate(zip(Y_sorted, X_sorted)):
    marker = '  <-- best' if i == 0 else ''
    x_str = ', '.join(f'{v:.6f}' for v in x_val)
    print(f'  [{i+1:2d}]  X = [{x_str}]   Y = {y_val:+.6e}{marker}')
print('=' * 72)

best_Y = Y_sorted[0]
best_X = np.array(X_sorted[0])
best_x_str = ', '.join(f'{v:.8f}' for v in best_X)
print(f'\n  Best Y*  = {best_Y:.6e}')
print(f'  Best X*  = [{best_x_str}]')

Input  shape : (15, 2)
Output shape : (15,)

  All observations (sorted descending by Y)
  [ 1]  X = [0.702637, 0.926564]   Y = +6.112052e-01  <-- best
  [ 2]  X = [0.665800, 0.123969]   Y = +5.389961e-01
  [ 3]  X = [0.877791, 0.778628]   Y = +4.205862e-01
  [ 4]  X = [0.845275, 0.711120]   Y = +2.939929e-01
  [ 5]  X = [0.438166, 0.685018]   Y = +2.446193e-01
  [ 6]  X = [0.454647, 0.290455]   Y = +2.149645e-01
  [ 7]  X = [1.000000, 0.326531]   Y = +1.492696e-01
  [ 8]  X = [0.341750, 0.028698]   Y = +3.874902e-02
  [ 9]  X = [1.000000, 1.000000]   Y = +3.643861e-02
  [10]  X = [0.001256, 0.001021]   Y = +3.251170e-02
  [11]  X = [0.000000, 1.000000]   Y = +2.486947e-02
  [12]  X = [0.577713, 0.771973]   Y = +2.310555e-02
  [13]  X = [0.338648, 0.213867]   Y = -1.385762e-02
  [14]  X = [0.998382, 0.000400]   Y = -5.480954e-02
  [15]  X = [0.142699, 0.349005]   Y = -6.562362e-02

  Best Y*  = 6.112052e-01
  Best X*  = [0.70263656, 0.92656420]


In [3]:
# Cell 4: Fit GP
# F2 is a NOISY function with values in normal range (-0.05 to +0.15).
# Fit on raw Y -- no log transform needed (values are not astronomically scaled).
# alpha=1e-4 allows noise in observations instead of forcing exact interpolation.
import warnings
warnings.filterwarnings('ignore')
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF

kernel = RBF(length_scale=0.1, length_scale_bounds='fixed')
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-4)
gp.fit(X, Y)

print('=' * 62)
print('  GP Fitting Results')
print('=' * 62)
print(f'  Kernel                  : {gp.kernel_}')
print(f'  Alpha (noise)           : 1e-4  (noisy function)')
print(f'  Log-marginal-likelihood : {gp.log_marginal_likelihood_value_:.4f}')
pred_mean, pred_std = gp.predict(best_X.reshape(1, -1), return_std=True)
print('  Sanity check at best known X*:')
print(f'    X*                     = [{best_X[0]:.6f}, {best_X[1]:.6f}]')
print(f'    GP predicted mean      = {pred_mean[0]:.6f}')
print(f'    Actual Y*              = {best_Y:.6f}')
print(f'    GP predicted std       = {pred_std[0]:.8f}')
print('=' * 62)

  GP Fitting Results
  Kernel                  : RBF(length_scale=0.1)
  Alpha (noise)           : 1e-4  (noisy function)
  Log-marginal-likelihood : -13.6973
  Sanity check at best known X*:
    X*                     = [0.702637, 0.926564]
    GP predicted mean      = 0.611145
    Actual Y*              = 0.611205
    GP predicted std       = 0.00999949


In [4]:
# Cell 5: GP UCB Acquisition -- CORRECTED grid around the true best region
# CRITICAL FIX: W5 grid [0.80,1.00] x [0.15,0.50] was centred on W2 weekly peak [1.0, 0.327] at Y=0.149.
# ACTUAL best: initial data point [0.703, 0.927] at Y=0.611 -- 4x better, completely outside old grid.
# NEW grid: x1:[0.60,0.80] x x2:[0.85,1.00] -- centred on the true best [0.703, 0.927].

x1 = np.linspace(0.60, 0.80, 100)
x2 = np.linspace(0.85, 1.00, 100)
xx1, xx2 = np.meshgrid(x1, x2)
X_grid = np.column_stack([xx1.ravel(), xx2.ravel()])

gp_mean, gp_std = gp.predict(X_grid, return_std=True)

# beta=1.0: exploitation -- exploit the known best region
beta = 1.0
ucb_scores = gp_mean + beta * gp_std

best_idx = np.argmax(ucb_scores)
next_x = X_grid[best_idx]
portal_string = f'{next_x[0]:.6f}-{next_x[1]:.6f}'

print('=' * 62)
print('  GP UCB Acquisition (beta=1.0, correct region)')
print('=' * 62)
print(f'  Search grid  : x1=[0.60,0.80] x x2=[0.85,1.00]  (10,000 pts)')
print(f'  Anchor       : initial best [0.703, 0.927] at Y=0.611')
print(f'  Beta         : {beta}  (exploitation)')
next_str = ', '.join(f'{v:.6f}' for v in next_x)
print(f'  Next query   : [{next_str}]')
print(f'  UCB score    : {ucb_scores[best_idx]:.6f}')
print(f'  GP mean      : {gp_mean[best_idx]:.6f}')
print(f'  GP std       : {gp_std[best_idx]:.6f}')
print('=' * 62)

  GP UCB Acquisition (beta=1.0, correct region)
  Search grid  : x1=[0.60,0.80] x x2=[0.85,1.00]  (10,000 pts)
  Anchor       : initial best [0.703, 0.927] at Y=0.611
  Beta         : 1.0  (exploitation)
  Next query   : [0.800000, 0.900000]
  UCB score    : 1.209261
  GP mean      : 0.494150
  GP std       : 0.715111


In [5]:
# Cell 6: Sanity check -- ensure query stays near the actual best [0.703, 0.927]

ZONE_X1_LO, ZONE_X1_HI = 0.60, 0.80
ZONE_X2_LO, ZONE_X2_HI = 0.85, 1.00

initial_best = np.array([0.702637, 0.926564])  # actual best, Y=0.611
dist_to_best = np.linalg.norm(next_x - initial_best)
in_zone = (ZONE_X1_LO <= next_x[0] <= ZONE_X1_HI) and (ZONE_X2_LO <= next_x[1] <= ZONE_X2_HI)

print('=' * 62)
print('  Sanity Check')
print('=' * 62)
print(f'  Proposed query   : [{next_x[0]:.6f}, {next_x[1]:.6f}]')
print(f'  Initial best X*  : [0.702637, 0.926564]  (Y=0.611)')
print(f'  Distance to X*   : {dist_to_best:.6f}')
print(f'  In zone          : {in_zone}  (x1:[0.60,0.80] x x2:[0.85,1.00])')
print()

if dist_to_best > 0.20:
    print(f'  WARNING: query is {dist_to_best:.4f} from initial best (> 0.20). Grid may be drifting.')
else:
    print(f'  OK: query is within 0.20 of initial best ({dist_to_best:.4f}).')

if not in_zone:
    print('  WARNING: query is outside x1:[0.60,0.80] x x2:[0.85,1.00]. Wrong region!')
else:
    print('  OK: query is inside the initial-best zone.')
print('=' * 62)

  Sanity Check
  Proposed query   : [0.800000, 0.900000]
  Initial best X*  : [0.702637, 0.926564]  (Y=0.611)
  Distance to X*   : 0.100922
  In zone          : True  (x1:[0.60,0.80] x x2:[0.85,1.00])

  OK: query is within 0.20 of initial best (0.1009).
  OK: query is inside the initial-best zone.


In [6]:
print('=' * 72)
print('  SUMMARY -- Week 6 Bayesian Optimisation')
print('=' * 72)
print('  Function        : 2 -- Noisy Chemical Process (2D)')
print('  Objective       : Maximise Y')
print(f'  Method          : GP UCB (beta=1.0, focused grid, raw Y fit)')
print()
best_x_s = ', '.join(f'{v:.6f}' for v in best_X)
next_s   = ', '.join(f'{v:.6f}' for v in next_x)
print(f'  Best X*         : [{best_x_s}]')
print(f'  Best Y*         : {best_Y:.6e}')
print()
print(f'  Next query      : [{next_s}]')
print()
print('  Portal submission string:')
print(f'  >>> {portal_string} <<<')
print('=' * 72)

  SUMMARY -- Week 6 Bayesian Optimisation
  Function        : 2 -- Noisy Chemical Process (2D)
  Objective       : Maximise Y
  Method          : GP UCB (beta=1.0, focused grid, raw Y fit)

  Best X*         : [0.702637, 0.926564]
  Best Y*         : 6.112052e-01

  Next query      : [0.800000, 0.900000]

  Portal submission string:
  >>> 0.800000-0.900000 <<<


## Week 6 -- Reflection

**Function**: F2 -- Noisy Chemical Process (2D)

### Strategy change from Week 5
- Removed SVM: it endorsed sending W5 to dead corner [0.001, 0.001] -- complete waste.
- No log transform: F2 values are in normal range (-0.05 to +0.15), log distorts the GP model.
- alpha=1e-4: function is noisy, GP should not force exact interpolation through every point.
- Sort by Y directly (not abs(Y)): positive = better for this function.
- Focused grid x1:[0.80,1.00] x x2:[0.15,0.50]: W2/W3/W4 bracket the peak at [~1.0, ~0.33].
- Beta=1.0: exploit the bracketed peak.

### Query
- **Input submitted**: *(fill in portal submission string)*
- **Output received**: *(fill in after result)*
- **Best so far**: *(update after result)*

### What this result taught us
*(fill in after receiving result)*

### Strategy for Week 7
*(fill in after reflecting on result)*